# Демо: публикуем модель на HuggingFace Hub

Прокликай по ячейкам сверху вниз — покажем, как опубликовать **готовую** модель.
Стартуем «с этого места»: модель уже обучена и лежит на Hub —
[HOhus/pushkin-nano-bpe](https://huggingface.co/HOhus/pushkin-nano-bpe) (из модуля 5).
Здесь только публикация; что такое форматы и локальный запуск (GGUF, MLX, LM Studio) —
в [уроке 5.1](https://itrubnikov.github.io/Train_of_Thought/docs/modules/05-1-publish-model/).

In [1]:
# Окружение + логин. В Colab pip уже почти всё ставит.
!pip install -q transformers huggingface_hub
import os
from huggingface_hub import login, whoami
# Залогинься: либо положи токен (scope WRITE) в переменную HF_TOKEN,
# либо запусти отдельно:  from huggingface_hub import notebook_login; notebook_login()
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])
HF_USER = whoami()["name"]
print("залогинен как:", HF_USER)


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


/Users/ivantrubnikov/Documents/claude/Train_of_Thought/git/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


залогинен как: HOhus


## Шаг 0. Берём готовую модель

Загрузим обученную модель с Hub в две строки — это и есть «нативный» артефакт.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
SRC = "HOhus/pushkin-nano-bpe"          # уже обученная модель из модуля 5
model = AutoModelForCausalLM.from_pretrained(SRC)
tok   = AutoTokenizer.from_pretrained(SRC)
print("взяли модель:", model.config.model_type, "| параметров:", model.num_parameters())

Loading weights: 100%|██████████| 40/40 [00:00<00:00, 14629.59it/s]


взяли модель: gpt2 | параметров: 742528


## Шаг 1. Из каких файлов состоит модель

`save_pretrained` кладёт веса (`model.safetensors`), конфиг (`config.json`) и токенизатор в папку.

In [3]:
# Из каких файлов состоит модель — сохраним в папку и посмотрим.
model.save_pretrained("my-model")
tok.save_pretrained("my-model")
print("файлы модели:")
for f in sorted(os.listdir("my-model")):
    print("  ", f, "-", os.path.getsize(os.path.join("my-model", f)), "bytes")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 192.95it/s]

файлы модели:
   config.json - 821 bytes
   generation_config.json - 195 bytes
   model.safetensors - 2973984 bytes
   tokenizer.json - 64354 bytes
   tokenizer_config.json - 305 bytes


## Шаг 2. Публикуем под своим именем

`push_to_hub` создаёт репозиторий и заливает файлы. Нужен токен со scope `write`.

In [4]:
REPO = f"{HF_USER}/pushkin-nano-bpe"   # станет huggingface.co/<твой-username>/pushkin-nano-bpe
model.push_to_hub(REPO)
tok.push_to_hub(REPO)
print("опубликовано:", "https://huggingface.co/" + REPO)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 586.53it/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 2.97MB / 2.97MB,  277kB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


опубликовано: https://huggingface.co/HOhus/pushkin-nano-bpe


## Шаг 3. Проверяем

Открой ссылку выше: там карточка, файлы (`model.safetensors`, `config.json`, токенизатор) и кнопка **Use this model**. И заберём модель обратно — докажем, что публикация рабочая:

In [5]:
back = AutoModelForCausalLM.from_pretrained(REPO)
print("скачали обратно с Hub, параметров:", back.num_parameters())

Loading weights: 100%|██████████| 40/40 [00:00<00:00, 10928.36it/s]


скачали обратно с Hub, параметров: 742528


## Шаг 4 (бонус). GGUF для LM Studio

LM Studio и Ollama понимают только **GGUF/MLX**, а не Safetensors. Сконвертируем
модель в GGUF и зальём отдельный репозиторий — тогда LM Studio её найдёт. Займёт
пару минут (клонируется llama.cpp). Подробнее про форматы — в
[уроке 5.1](https://itrubnikov.github.io/Train_of_Thought/docs/modules/05-1-publish-model/).

In [ ]:
# инструменты + опубликованная модель локально
!pip install -q gguf
![ -d llama.cpp ] || git clone --depth 1 https://github.com/ggerganov/llama.cpp
from huggingface_hub import snapshot_download
snapshot_download(REPO, local_dir="pushkin-bpe-hf")   # REPO из Шага 2
print("готово к конвертации")

In [ ]:
# Две правки под нашу игрушечную модель (для «взрослых» моделей не нужны):
import json, glob
# 1) свежий transformers убрал n_ctx из GPT2Config -> вернём (= n_positions)
cfg = json.load(open("pushkin-bpe-hf/config.json"))
cfg["n_ctx"] = cfg.get("n_positions", 128)
json.dump(cfg, open("pushkin-bpe-hf/config.json", "w"), indent=2)
# 2) llama.cpp не узнаёт наш кастомный BPE по хешу — он byte-level GPT-2-стиля,
#    поэтому просим конвертер считать его "gpt-2"
anchor = 'raise NotImplementedError("BPE pre-tokenizer was not recognized - update get_vocab_base_pre()")'
for f in glob.glob("llama.cpp/**/*.py", recursive=True):
    s = open(f).read()
    if anchor in s:
        open(f, "w").write(s.replace(anchor, 'res = "gpt-2"'))
        print("патч претокенайзера ->", f)

In [ ]:
# Конвертация в GGUF (sys.executable — чтобы работало и в Colab, и локально)
import sys, subprocess
subprocess.run([sys.executable, "llama.cpp/convert_hf_to_gguf.py", "pushkin-bpe-hf",
                "--outfile", "pushkin-nano-bpe.gguf", "--outtype", "f16"], check=True)

# Заливаем .gguf в ОТДЕЛЬНЫЙ репозиторий — LM Studio ищет именно GGUF/MLX
from huggingface_hub import HfApi, create_repo
GGUF_REPO = f"{HF_USER}/pushkin-nano-bpe-GGUF"
create_repo(GGUF_REPO, exist_ok=True)
HfApi().upload_file(path_or_fileobj="pushkin-nano-bpe.gguf",
                    path_in_repo="pushkin-nano-bpe.gguf", repo_id=GGUF_REPO)
print("GGUF на Hub:", "https://huggingface.co/" + GGUF_REPO)

Теперь в **LM Studio**: Model Search → `<твой-username>/pushkin-nano-bpe-GGUF`
(галочка GGUF). Или положи файл в `~/.lmstudio/models/<username>/pushkin-nano-bpe-GGUF/`
— и он появится в **My Models** без поиска.

Текст будет абракадаброй: модель игрушечная (0.7 млн параметров, одна эпоха).
Ценность — путь «обучил -> опубликовал -> GGUF -> локальный запуск» сработал целиком.

---

**Итог:** собрать файлы → `push_to_hub` → модель на Hub, её заберёт любой одной командой.
Дальше в [уроке 5.1](https://itrubnikov.github.io/Train_of_Thought/docs/modules/05-1-publish-model/) —
форматы (GGUF/MLX) и локальный запуск в LM Studio.